In [ ]:
import json
import os
import platform
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

IN_COLAB = False
try:
    import google.colab  
    from google.colab import drive  
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = "/content/drive"
    MOUNT_POINT = f"{DRIVE_ROOT}/MyDrive"
    try:
        if not os.path.exists(MOUNT_POINT):
            drive.mount(DRIVE_ROOT, force_remount=True)
        else:
            drive.mount(DRIVE_ROOT, force_remount=False)
    except Exception as e:
        print(f"[WARN] Drive mount issue: {e}")
    BASE_ROOT = Path(MOUNT_POINT) / "Outputs"
else:
    BASE_ROOT = Path("Outputs")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_ROOT = BASE_ROOT / f"bb84_finite_key_phase_study_{RUN_ID}"
FIG_DIR = PROJECT_ROOT / "figures"
TAB_DIR = PROJECT_ROOT / "tables"
OTH_DIR = PROJECT_ROOT / "others"

for directory in (FIG_DIR, TAB_DIR, OTH_DIR):
    directory.mkdir(parents=True, exist_ok=True)

run_manifest = {
    "run_id": RUN_ID,
    "seed": SEED,
    "in_colab": IN_COLAB,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "project_root": str(PROJECT_ROOT),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TAB_DIR),
    "other_dir": str(OTH_DIR),
}

manifest_path = OTH_DIR / "run_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("[ENV] Colab:", IN_COLAB)
print("[RUN]", RUN_ID)
print("[SEED]", SEED)
print("[ROOT]", PROJECT_ROOT)
print("[MANIFEST]", manifest_path)
print(
    "[VERSIONS]",
    {
        "python": run_manifest["python_version"],
        "numpy": run_manifest["numpy_version"],
        "pandas": run_manifest["pandas_version"],
        "matplotlib": run_manifest["matplotlib_version"],
    },
)

OKABE_ITO = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # bluish green
    "#CC79A7",  # reddish purple
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#F0E442",  # yellow
    "#000000",  # black
]

LINESTYLES = [
    "-",
    "--",
    "-.",
    ":",
    (0, (5, 2)),
    (0, (3, 1, 1, 1)),
    (0, (1, 1)),
    (0, (3, 1, 1, 1, 1, 1)),
]

MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

BAND_ALPHA = 0.18


def series_color(index):
    return OKABE_ITO[int(index) % len(OKABE_ITO)]


def series_style(index, lw=1.8, ms=4.5, marker=True):
    """Redundant colour + line-style + marker encoding for series `index`."""
    i = int(index)
    style = {
        "color": OKABE_ITO[i % len(OKABE_ITO)],
        "linestyle": LINESTYLES[i % len(LINESTYLES)],
        "linewidth": lw,
    }
    if marker:
        style["marker"] = MARKERS[i % len(MARKERS)]
        style["markersize"] = ms
    return style


plt.rcParams.update(
    {
        "axes.prop_cycle": matplotlib.cycler(color=OKABE_ITO),
        "figure.dpi": 110,
        "savefig.dpi": 240,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "legend.framealpha": 0.9,
        "font.size": 10.5,
    }
)

run_manifest["palette"] = "Okabe-Ito (colour-blind safe)"
run_manifest["redundant_encoding"] = "colour + line style + marker"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("[STYLE] palette:", run_manifest["palette"])
print("[STYLE] encoding:", run_manifest["redundant_encoding"])


[ENV] Colab: False
[RUN] 20260824_225821
[SEED] 42
[ROOT] Outputs\bb84_finite_key_phase_study_20260824_225821
[MANIFEST] Outputs\bb84_finite_key_phase_study_20260824_225821\others\run_manifest.json
[VERSIONS] {'python': '3.10.20', 'numpy': '2.2.5', 'pandas': '2.3.3', 'matplotlib': '3.10.8'}
[STYLE] palette: Okabe-Ito (colour-blind safe)
[STYLE] encoding: colour + line style + marker


In [2]:
TIME_HORIZON = 120
T = np.arange(TIME_HORIZON, dtype=float)

EPS = 1e-12

QBER_SECURITY_THRESHOLD = 0.11
PHASE_MARGIN = 0.02

ETA_GRID = np.linspace(0.01, 0.22, 85)
ATTACK_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])
DRIFT_GRID = np.array([0.005, 0.010, 0.020, 0.030])

SHOT_COUNTS = np.array([2_000, 5_000, 10_000, 20_000])
DEFAULT_SHOTS = 10_000
N_REPEATS = 250

SCENARIOS = [
    {"name": "low_disturbance", "eta": 0.035, "attack": 0.08, "drift": 0.008},
    {"name": "transition_margin", "eta": 0.075, "attack": 0.16, "drift": 0.015},
    {"name": "elevated_pressure", "eta": 0.105, "attack": 0.24, "drift": 0.020},
    {"name": "high_disturbance", "eta": 0.145, "attack": 0.32, "drift": 0.030},
]

BB84_STAGES = [
    "State preparation",
    "Quantum transmission",
    "Sifting and basis reconciliation",
    "Parameter estimation",
    "Error correction",
    "Privacy amplification",
]

BB84_STAGE_MAP = {
    "State preparation": "Source fidelity and preparation stability.",
    "Quantum transmission": "Channel disturbance and intercept pressure.",
    "Sifting and basis reconciliation": "Observed mismatch after basis filtering.",
    "Parameter estimation": "Finite-key inference for QBER and secrecy margin.",
    "Error correction": "Leakage burden induced by observed disturbance.",
    "Privacy amplification": "Residual extractable secrecy after finite-key penalties.",
}

print("[CONFIG] time horizon:", TIME_HORIZON)
print("[CONFIG] eta range:", (float(ETA_GRID.min()), float(ETA_GRID.max())))
print("[CONFIG] attack levels:", ATTACK_GRID.tolist())
print("[CONFIG] drift levels:", DRIFT_GRID.tolist())
print("[CONFIG] shot counts:", SHOT_COUNTS.tolist())
print("[CONFIG] repeats:", N_REPEATS)
print("[CONFIG] qber threshold:", QBER_SECURITY_THRESHOLD)
print("[BB84] stages:", " | ".join(BB84_STAGES))

[CONFIG] time horizon: 120
[CONFIG] eta range: (0.01, 0.22)
[CONFIG] attack levels: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
[CONFIG] drift levels: [0.005, 0.01, 0.02, 0.03]
[CONFIG] shot counts: [2000, 5000, 10000, 20000]
[CONFIG] repeats: 250
[CONFIG] qber threshold: 0.11
[BB84] stages: State preparation | Quantum transmission | Sifting and basis reconciliation | Parameter estimation | Error correction | Privacy amplification


In [ ]:

COEFF_BASELINE = {
    "w_a": 0.055,
    "w_d": 0.90,
    "r_a": 0.18,
    "r_d": 1.10,
    "kappa": 2.6,
    "w_M": 1.35,
}

COEFFS = dict(COEFF_BASELINE)


def reset_coefficients():
    """Restore all structural coefficients to their baseline values."""
    COEFFS.update(COEFF_BASELINE)


def clip01(x):
    return np.clip(x, 0.0, 1.0)


def binary_entropy(p):
    p = np.clip(p, EPS, 1.0 - EPS)
    return -(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p))


def deterministic_qber_mean(eta, attack, drift, t, time_horizon=TIME_HORIZON):
    seasonal = 0.006 * np.sin(2.0 * np.pi * t / max(time_horizon, 1))
    trend = 0.010 * (t / max(time_horizon - 1, 1))
    qber_mean = eta + COEFFS["w_a"] * attack + COEFFS["w_d"] * drift + seasonal + trend
    return float(clip01(qber_mean))


def effective_detection_count(shots, attack, drift):
    retention = 1.0 - COEFFS["r_a"] * attack - COEFFS["r_d"] * drift
    retention = float(np.clip(retention, 0.35, 1.0))
    return max(50, int(round(shots * retention)))


def sample_observed_qber(qber_mean, shots, attack, drift, rng):
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)
    errors = rng.binomial(n=n_eff, p=float(np.clip(qber_mean, EPS, 1.0 - EPS)))
    return float(errors / n_eff), n_eff


def finite_key_penalty(shots):
    return float(COEFFS["kappa"] / np.sqrt(max(shots, 1)))


def secrecy_margin(qber_obs, shots, drift):
    margin = (
        1.0
        - 2.0 * binary_entropy(np.minimum(qber_obs, 0.499999))
        - finite_key_penalty(shots)
        - COEFFS["w_M"] * drift
    )
    return float(margin)


def leakage_burden(qber_obs, attack, drift):
    leakage = 0.65 * attack + 0.75 * qber_obs + 0.90 * drift
    return float(leakage)


def order_parameter_from_margin(margin):
    scale = max(abs(QBER_SECURITY_THRESHOLD), EPS)
    return float(margin / scale)


def classify_phase(psi, margin=PHASE_MARGIN):
    if psi > margin:
        return "secure"
    if psi < -margin:
        return "insecure"
    return "transition"


def susceptibility(values, controls):
    return np.gradient(np.asarray(values, dtype=float), np.asarray(controls, dtype=float))


def first_transition_time(psi_series):
    idx = np.where(np.asarray(psi_series) <= 0.0)[0]
    return int(idx[0]) if len(idx) else -1


rng_smoke = np.random.default_rng(SEED)
qber_mean_smoke = deterministic_qber_mean(eta=0.06, attack=0.10, drift=0.01, t=15)
qber_obs_smoke, n_eff_smoke = sample_observed_qber(
    qber_mean_smoke,
    shots=DEFAULT_SHOTS,
    attack=0.10,
    drift=0.01,
    rng=rng_smoke,
)
margin_smoke = secrecy_margin(qber_obs_smoke, shots=n_eff_smoke, drift=0.01)
psi_smoke = order_parameter_from_margin(margin_smoke)

print("[COEFF] baseline:", COEFF_BASELINE)
print("[SMOKE] mean qber =", round(qber_mean_smoke, 6))
print("[SMOKE] observed qber =", round(qber_obs_smoke, 6))
print("[SMOKE] effective detections =", n_eff_smoke)
print("[SMOKE] secrecy margin =", round(margin_smoke, 6))
print("[SMOKE] psi =", round(psi_smoke, 6))
print("[SMOKE] phase =", classify_phase(psi_smoke))


[COEFF] baseline: {'w_a': 0.055, 'w_d': 0.9, 'r_a': 0.18, 'r_d': 1.1, 'kappa': 2.6, 'w_M': 1.35}
[SMOKE] mean qber = 0.080003
[SMOKE] observed qber = 0.077137
[SMOKE] effective detections = 9710
[SMOKE] secrecy margin = 0.176095
[SMOKE] psi = 1.600862
[SMOKE] phase = secure


In [4]:
def simulate_trajectory(eta, attack, drift, shots=DEFAULT_SHOTS, time_horizon=TIME_HORIZON, seed=None):
    rng = np.random.default_rng(SEED if seed is None else seed)

    tt = np.arange(time_horizon, dtype=float)
    qber_mean = np.zeros(time_horizon, dtype=float)
    qber_obs = np.zeros(time_horizon, dtype=float)
    n_eff = np.zeros(time_horizon, dtype=int)
    margin = np.zeros(time_horizon, dtype=float)
    leakage = np.zeros(time_horizon, dtype=float)

    for k in range(time_horizon):
        qber_mean[k] = deterministic_qber_mean(
            eta=eta,
            attack=attack,
            drift=drift,
            t=k,
            time_horizon=time_horizon,
        )
        qber_obs[k], n_eff[k] = sample_observed_qber(
            qber_mean=qber_mean[k],
            shots=shots,
            attack=attack,
            drift=drift,
            rng=rng,
        )
        margin[k] = secrecy_margin(qber_obs=qber_obs[k], shots=n_eff[k], drift=drift)
        leakage[k] = leakage_burden(qber_obs=qber_obs[k], attack=attack, drift=drift)

    psi = np.array([order_parameter_from_margin(x) for x in margin], dtype=float)
    dpsi_dt = np.gradient(psi, tt)
    dqber_dt = np.gradient(qber_obs, tt)

    return pd.DataFrame(
        {
            "t": tt,
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "n_eff": n_eff,
            "qber_mean": qber_mean,
            "qber_obs": qber_obs,
            "secrecy_margin": margin,
            "leakage_burden": leakage,
            "psi": psi,
            "dpsi_dt": dpsi_dt,
            "dqber_dt": dqber_dt,
            "phase": [classify_phase(x) for x in psi],
        }
    )


smoke_df = simulate_trajectory(
    eta=0.07,
    attack=0.14,
    drift=0.012,
    shots=DEFAULT_SHOTS,
    seed=SEED,
)

print(smoke_df.head(3).to_string(index=False))
print("[SMOKE] first transition time =", first_transition_time(smoke_df["psi"].values))
print("[SMOKE] final phase =", smoke_df["phase"].iloc[-1])

  t  eta  attack  drift  shots  n_eff  qber_mean  qber_obs  secrecy_margin  leakage_burden      psi   dpsi_dt  dqber_dt  phase
0.0 0.07    0.14  0.012  10000   9616   0.088500  0.085483        0.114869        0.165912 1.044262 -0.287319  0.004680 secure
1.0 0.07    0.14  0.012  10000   9616   0.088898  0.090162        0.083264        0.169422 0.756943  0.042178 -0.000676 secure
2.0 0.07    0.14  0.012  10000   9616   0.089295  0.084131        0.124148        0.164898 1.128618  0.066595 -0.001092 secure
[SMOKE] first transition time = -1
[SMOKE] final phase = secure


In [5]:
def empirical_interval(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    lo, med, hi = np.quantile(values, [alpha / 2.0, 0.5, 1.0 - alpha / 2.0])
    return float(lo), float(med), float(hi)


def simulate_endpoint_replicates(
    eta,
    attack,
    drift,
    shots=DEFAULT_SHOTS,
    n_repeats=N_REPEATS,
    t_eval=TIME_HORIZON - 1,
    rng=None,
):
    rng = np.random.default_rng(SEED) if rng is None else rng

    qber_mean = deterministic_qber_mean(
        eta=eta,
        attack=attack,
        drift=drift,
        t=t_eval,
        time_horizon=TIME_HORIZON,
    )
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)
    errors = rng.binomial(
        n=n_eff,
        p=float(np.clip(qber_mean, EPS, 1.0 - EPS)),
        size=n_repeats,
    )
    qber_obs = errors / n_eff
    margins = np.array([secrecy_margin(q, shots=n_eff, drift=drift) for q in qber_obs], dtype=float)
    leakage = np.array([leakage_burden(q, attack=attack, drift=drift) for q in qber_obs], dtype=float)
    psi = np.array([order_parameter_from_margin(m) for m in margins], dtype=float)
    phase = np.array([classify_phase(x) for x in psi], dtype=object)

    return pd.DataFrame(
        {
            "eta": float(eta),
            "attack": float(attack),
            "drift": float(drift),
            "shots": int(shots),
            "t_eval": int(t_eval),
            "repeat": np.arange(n_repeats, dtype=int),
            "n_eff": int(n_eff),
            "qber_mean": float(qber_mean),
            "qber_obs": qber_obs,
            "secrecy_margin": margins,
            "leakage_burden": leakage,
            "psi": psi,
            "phase": phase,
        }
    )


rng_panel = np.random.default_rng(SEED)
endpoint_frames = []
summary_rows = []

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        for eta in ETA_GRID:
            df = simulate_endpoint_replicates(
                eta=float(eta),
                attack=float(attack),
                drift=float(drift),
                shots=DEFAULT_SHOTS,
                n_repeats=N_REPEATS,
                rng=rng_panel,
            )
            endpoint_frames.append(df)

            qber_lo, qber_med, qber_hi = empirical_interval(df["qber_obs"].values)
            margin_lo, margin_med, margin_hi = empirical_interval(df["secrecy_margin"].values)
            psi_lo, psi_med, psi_hi = empirical_interval(df["psi"].values)

            phase_counts = df["phase"].value_counts()
            p_secure = float((df["phase"] == "secure").mean())
            p_transition = float((df["phase"] == "transition").mean())
            p_insecure = float((df["phase"] == "insecure").mean())
            dominant_phase = phase_counts.idxmax()

            summary_rows.append(
                {
                    "eta": float(eta),
                    "attack": float(attack),
                    "drift": float(drift),
                    "shots": int(DEFAULT_SHOTS),
                    "n_repeats": int(N_REPEATS),
                    "n_eff": int(df["n_eff"].iloc[0]),
                    "qber_mean_det": float(df["qber_mean"].iloc[0]),
                    "qber_obs_mean": float(df["qber_obs"].mean()),
                    "qber_obs_sd": float(df["qber_obs"].std(ddof=1)),
                    "qber_obs_ci_low": qber_lo,
                    "qber_obs_ci_med": qber_med,
                    "qber_obs_ci_high": qber_hi,
                    "margin_mean": float(df["secrecy_margin"].mean()),
                    "margin_sd": float(df["secrecy_margin"].std(ddof=1)),
                    "margin_ci_low": margin_lo,
                    "margin_ci_med": margin_med,
                    "margin_ci_high": margin_hi,
                    "psi_mean": float(df["psi"].mean()),
                    "psi_sd": float(df["psi"].std(ddof=1)),
                    "psi_ci_low": psi_lo,
                    "psi_ci_med": psi_med,
                    "psi_ci_high": psi_hi,
                    "mean_leakage": float(df["leakage_burden"].mean()),
                    "p_secure": p_secure,
                    "p_transition": p_transition,
                    "p_insecure": p_insecure,
                    "dominant_phase": dominant_phase,
                }
            )

endpoint_df = pd.concat(endpoint_frames, ignore_index=True)
endpoint_summary_df = pd.DataFrame(summary_rows).sort_values(["drift", "attack", "eta"]).reset_index(drop=True)

endpoint_csv = TAB_DIR / "endpoint_replicates.csv"
endpoint_summary_csv = TAB_DIR / "endpoint_summary.csv"

endpoint_df.to_csv(endpoint_csv, index=False)
endpoint_summary_df.to_csv(endpoint_summary_csv, index=False)

print("[ENDPOINT] replicate rows =", len(endpoint_df))
print("[ENDPOINT] summary rows =", len(endpoint_summary_df))
print("[TABLE] saved:", endpoint_csv)
print("[TABLE] saved:", endpoint_summary_csv)
print(endpoint_summary_df.head(8).to_string(index=False))

[ENDPOINT] replicate rows = 680000
[ENDPOINT] summary rows = 2720
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\endpoint_replicates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\endpoint_summary.csv
   eta  attack  drift  shots  n_repeats  n_eff  qber_mean_det  qber_obs_mean  qber_obs_sd  qber_obs_ci_low  qber_obs_ci_med  qber_obs_ci_high  margin_mean  margin_sd  margin_ci_low  margin_ci_med  margin_ci_high  psi_mean   psi_sd  psi_ci_low  psi_ci_med  psi_ci_high  mean_leakage  p_secure  p_transition  p_insecure dominant_phase
0.0100    0.05  0.005  10000        250   9855       0.026936       0.026748     0.001646         0.023384         0.026738          0.029934     0.611588   0.017093       0.578937       0.611544        0.646978  5.559894 0.155391    5.263068    5.559488     5.881615      0.057061       1.0           0.0         0.0         secure
0.0125    0.05  0.005  10000        250   9855       0.029436       0.02944

In [6]:
def interpolate_crossing(x, y, target):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float) - float(target)

    exact = np.where(np.isclose(y, 0.0, atol=1e-12))[0]
    if len(exact):
        return float(x[int(exact[0])])

    crossing_idx = np.where(y[:-1] * y[1:] < 0.0)[0]
    if len(crossing_idx) == 0:
        return np.nan

    i = int(crossing_idx[0])
    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]
    return float(x0 - y0 * (x1 - x0) / (y1 - y0))


def bootstrap_eta_c(secure_matrix, eta_values, n_boot=200, seed=SEED):
    secure_matrix = np.asarray(secure_matrix, dtype=float)
    eta_values = np.asarray(eta_values, dtype=float)

    n_eta, n_rep = secure_matrix.shape
    rng = np.random.default_rng(seed)
    eta_boot = []

    row_index = np.arange(n_eta)[:, None]

    for _ in range(n_boot):
        draw_index = rng.integers(0, n_rep, size=(n_eta, n_rep))
        p_secure_boot = secure_matrix[row_index, draw_index].mean(axis=1)
        eta_c_boot = interpolate_crossing(eta_values, p_secure_boot, target=0.5)
        if np.isfinite(eta_c_boot):
            eta_boot.append(float(eta_c_boot))

    if len(eta_boot) == 0:
        return np.nan, np.nan, np.nan, 0

    eta_boot = np.asarray(eta_boot, dtype=float)
    lo, med, hi = np.quantile(eta_boot, [0.025, 0.5, 0.975])
    return float(lo), float(med), float(hi), int(len(eta_boot))


boundary_rows = []
group_seed_rng = np.random.default_rng(SEED)

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        sub = endpoint_summary_df[
            (endpoint_summary_df["drift"] == float(drift)) &
            (endpoint_summary_df["attack"] == float(attack))
        ].sort_values("eta")

        eta_values = sub["eta"].to_numpy(dtype=float)
        p_secure_values = sub["p_secure"].to_numpy(dtype=float)
        margin_values = sub["margin_mean"].to_numpy(dtype=float)

        eta_c_margin = interpolate_crossing(eta_values, margin_values, target=0.0)
        eta_c_psecure50 = interpolate_crossing(eta_values, p_secure_values, target=0.5)

        secure_rows = []
        for eta in eta_values:
            reps = endpoint_df[
                (endpoint_df["drift"] == float(drift)) &
                (endpoint_df["attack"] == float(attack)) &
                (endpoint_df["eta"] == float(eta))
            ].sort_values("repeat")
            secure_rows.append((reps["phase"] == "secure").to_numpy(dtype=float))

        secure_matrix = np.vstack(secure_rows)
        boot_seed = int(group_seed_rng.integers(0, 2**32 - 1))
        eta_c_low, eta_c_med, eta_c_high, n_boot_valid = bootstrap_eta_c(
            secure_matrix=secure_matrix,
            eta_values=eta_values,
            n_boot=200,
            seed=boot_seed,
        )

        dp_deta = susceptibility(p_secure_values, eta_values)
        eta_at_max_slope = float(eta_values[int(np.argmax(np.abs(dp_deta)))])

        boundary_rows.append(
            {
                "attack": float(attack),
                "drift": float(drift),
                "eta_c_margin": eta_c_margin,
                "eta_c_psecure50": eta_c_psecure50,
                "eta_c_boot_low": eta_c_low,
                "eta_c_boot_med": eta_c_med,
                "eta_c_boot_high": eta_c_high,
                "n_boot_valid": int(n_boot_valid),
                "p_secure_min_eta": float(p_secure_values[0]),
                "p_secure_max_eta": float(p_secure_values[-1]),
                "max_abs_dp_deta": float(np.max(np.abs(dp_deta))),
                "eta_at_max_abs_dp_deta": eta_at_max_slope,
            }
        )

boundary_df = pd.DataFrame(boundary_rows).sort_values(["drift", "attack"]).reset_index(drop=True)

boundary_csv = TAB_DIR / "phase_boundary_summary.csv"
boundary_df.to_csv(boundary_csv, index=False)

plt.figure(figsize=(8, 5.2))
for i, drift in enumerate(DRIFT_GRID):
    sub = boundary_df[boundary_df["drift"] == float(drift)].sort_values("attack")
    x = sub["attack"].to_numpy(dtype=float)
    y = sub["eta_c_boot_med"].to_numpy(dtype=float)
    ylo = sub["eta_c_boot_low"].to_numpy(dtype=float)
    yhi = sub["eta_c_boot_high"].to_numpy(dtype=float)

    plt.plot(x, y, label=f"drift={drift:.3f}", **series_style(i))
    plt.fill_between(x, ylo, yhi, color=series_color(i), alpha=BAND_ALPHA)

plt.xlabel("Attack pressure")
plt.ylabel(r"Estimated boundary $\eta_c$")
plt.title("Operational phase boundary with bootstrap uncertainty")
plt.legend()
plt.tight_layout()

boundary_fig = FIG_DIR / "phase_boundary_with_uncertainty.png"
plt.savefig(boundary_fig, dpi=240)
plt.close()

print("[BOUNDARY] rows =", len(boundary_df))
print("[TABLE] saved:", boundary_csv)
print("[FIGURE] saved:", boundary_fig)
print(boundary_df.head(10).to_string(index=False))

[BOUNDARY] rows = 32
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\phase_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\phase_boundary_with_uncertainty.png
 attack  drift  eta_c_margin  eta_c_psecure50  eta_c_boot_low  eta_c_boot_med  eta_c_boot_high  n_boot_valid  p_secure_min_eta  p_secure_max_eta  max_abs_dp_deta  eta_at_max_abs_dp_deta
   0.05  0.005      0.087717         0.087237        0.086738        0.087252         0.087781           200               1.0               0.0            104.0                  0.0875
   0.10  0.005      0.084732         0.084412        0.084133        0.084427         0.084922           200               1.0               0.0            113.6                  0.0850
   0.15  0.005      0.082298         0.082045        0.081699        0.082055         0.082466           200               1.0               0.0            126.4                  0.0825
   0.20  0.005      0.

In [7]:
profile_specs = [
    {"attack": 0.10, "drift": 0.005},
    {"attack": 0.20, "drift": 0.010},
    {"attack": 0.30, "drift": 0.020},
    {"attack": 0.35, "drift": 0.030},
]

profile_rows = []

for spec in profile_specs:
    attack = float(spec["attack"])
    drift = float(spec["drift"])

    sub = endpoint_summary_df[
        (endpoint_summary_df["attack"] == attack) &
        (endpoint_summary_df["drift"] == drift)
    ].sort_values("eta").copy()

    sub["dp_secure_deta"] = susceptibility(sub["p_secure"].values, sub["eta"].values)
    sub["profile_attack"] = attack
    sub["profile_drift"] = drift
    profile_rows.append(sub)

profiles_df = pd.concat(profile_rows, ignore_index=True)
profiles_csv = TAB_DIR / "transition_profiles.csv"
profiles_df.to_csv(profiles_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for i, drift in enumerate(DRIFT_GRID):
    attack = 0.20
    sub = endpoint_summary_df[
        (endpoint_summary_df["attack"] == attack) &
        (endpoint_summary_df["drift"] == float(drift))
    ].sort_values("eta")

    boundary_row = boundary_df[
        (boundary_df["attack"] == attack) &
        (boundary_df["drift"] == float(drift))
    ].iloc[0]

    axes[0].plot(
        sub["eta"],
        sub["p_secure"],
        label=f"drift={drift:.3f}",
        **series_style(i, ms=3.5),
    )
    axes[0].axvline(
        boundary_row["eta_c_boot_med"],
        color=series_color(i),
        linestyle=LINESTYLES[i % len(LINESTYLES)],
        linewidth=1.2,
        alpha=0.7,
    )

axes[0].set_xlabel(r"Disturbance $\eta$")
axes[0].set_ylabel("Secure-state probability")
axes[0].set_title(r"Secure-state probability profiles at fixed attack ($a=0.20$)")
axes[0].legend()

for i, spec in enumerate(profile_specs):
    attack = float(spec["attack"])
    drift = float(spec["drift"])

    sub = profiles_df[
        (profiles_df["profile_attack"] == attack) &
        (profiles_df["profile_drift"] == drift)
    ].sort_values("eta")

    axes[1].plot(
        sub["eta"],
        np.abs(sub["dp_secure_deta"]),
        label=f"a={attack:.2f}, d={drift:.3f}",
        **series_style(i, lw=2.0, marker=False),
    )

axes[1].set_xlabel(r"Disturbance $\eta$")
axes[1].set_ylabel(r"$\left|\partial p_{\mathrm{secure}} / \partial \eta\right|$")
axes[1].set_title("Transition sharpness across representative settings")
axes[1].legend()

plt.tight_layout()

profiles_fig = FIG_DIR / "transition_profiles_and_susceptibility.png"
plt.savefig(profiles_fig, dpi=240)
plt.close()

print("[PROFILE] rows =", len(profiles_df))
print("[TABLE] saved:", profiles_csv)
print("[FIGURE] saved:", profiles_fig)
print(
    profiles_df[
        [
            "profile_attack",
            "profile_drift",
            "eta",
            "p_secure",
            "dp_secure_deta",
            "dominant_phase",
        ]
    ].head(12).to_string(index=False)
)

[PROFILE] rows = 340
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\transition_profiles.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\transition_profiles_and_susceptibility.png
 profile_attack  profile_drift    eta  p_secure  dp_secure_deta dominant_phase
            0.1          0.005 0.0100       1.0             0.0         secure
            0.1          0.005 0.0125       1.0             0.0         secure
            0.1          0.005 0.0150       1.0             0.0         secure
            0.1          0.005 0.0175       1.0             0.0         secure
            0.1          0.005 0.0200       1.0             0.0         secure
            0.1          0.005 0.0225       1.0             0.0         secure
            0.1          0.005 0.0250       1.0             0.0         secure
            0.1          0.005 0.0275       1.0             0.0         secure
            0.1          0.005 0.0300       1.0    

In [8]:
shot_boundary_rows = []

for shots in SHOT_COUNTS:
    shot_seed_rng = np.random.default_rng(SEED + int(shots))
    shot_summary_rows = []

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            for eta in ETA_GRID:
                df = simulate_endpoint_replicates(
                    eta=float(eta),
                    attack=float(attack),
                    drift=float(drift),
                    shots=int(shots),
                    n_repeats=N_REPEATS,
                    rng=shot_seed_rng,
                )

                shot_summary_rows.append(
                    {
                        "eta": float(eta),
                        "attack": float(attack),
                        "drift": float(drift),
                        "shots": int(shots),
                        "p_secure": float((df["phase"] == "secure").mean()),
                        "margin_mean": float(df["secrecy_margin"].mean()),
                    }
                )

    shot_summary_df = (
        pd.DataFrame(shot_summary_rows)
        .sort_values(["drift", "attack", "eta"])
        .reset_index(drop=True)
    )

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = shot_summary_df[
                (shot_summary_df["drift"] == float(drift)) &
                (shot_summary_df["attack"] == float(attack))
            ].sort_values("eta")

            eta_values = sub["eta"].to_numpy(dtype=float)
            p_secure_values = sub["p_secure"].to_numpy(dtype=float)
            margin_values = sub["margin_mean"].to_numpy(dtype=float)

            eta_c_margin = interpolate_crossing(eta_values, margin_values, target=0.0)
            eta_c_psecure50 = interpolate_crossing(eta_values, p_secure_values, target=0.5)

            shot_boundary_rows.append(
                {
                    "shots": int(shots),
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_margin": eta_c_margin,
                    "eta_c_psecure50": eta_c_psecure50,
                }
            )

shot_boundary_df = (
    pd.DataFrame(shot_boundary_rows)
    .sort_values(["shots", "drift", "attack"])
    .reset_index(drop=True)
)

shot_boundary_csv = TAB_DIR / "shot_count_boundary_summary.csv"
shot_boundary_df.to_csv(shot_boundary_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

series_index = 0
for drift in [0.005, 0.020]:
    sub = shot_boundary_df[shot_boundary_df["drift"] == drift]
    for shots in SHOT_COUNTS:
        ssub = sub[sub["shots"] == int(shots)].sort_values("attack")
        axes[0].plot(
            ssub["attack"],
            ssub["eta_c_psecure50"],
            label=f"{shots} shots, drift={drift:.3f}",
            **series_style(series_index, ms=4.0),
        )
        series_index += 1

axes[0].set_xlabel("Attack pressure")
axes[0].set_ylabel(r"Estimated boundary $\eta_c$")
axes[0].set_title("Boundary location versus shot count")
axes[0].legend(fontsize=9)

agg = (
    shot_boundary_df
    .groupby("shots", as_index=False)
    .agg(
        eta_c_mean=("eta_c_psecure50", "mean"),
        eta_c_sd=("eta_c_psecure50", "std"),
    )
)

axes[1].errorbar(
    agg["shots"],
    agg["eta_c_mean"],
    yerr=agg["eta_c_sd"],
    capsize=4,
    ecolor=series_color(0),
    **series_style(0),
)
axes[1].set_xlabel("Shot count")
axes[1].set_ylabel(r"Mean estimated boundary $\eta_c$")
axes[1].set_title("Global boundary trend across finite sample sizes")

plt.tight_layout()

shot_boundary_fig = FIG_DIR / "shot_count_boundary_dependence.png"
plt.savefig(shot_boundary_fig, dpi=240)
plt.close()

print("[SHOT] rows =", len(shot_boundary_df))
print("[TABLE] saved:", shot_boundary_csv)
print("[FIGURE] saved:", shot_boundary_fig)
print(shot_boundary_df.head(12).to_string(index=False))
print(agg.to_string(index=False))

[SHOT] rows = 128
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\shot_count_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\shot_count_boundary_dependence.png
 shots  attack  drift  eta_c_margin  eta_c_psecure50
  2000    0.05  0.005      0.083020         0.082727
  2000    0.10  0.005      0.080332         0.079342
  2000    0.15  0.005      0.076978         0.076694
  2000    0.20  0.005      0.074493         0.073906
  2000    0.25  0.005      0.071132         0.070278
  2000    0.30  0.005      0.068506         0.068590
  2000    0.35  0.005      0.065571         0.065278
  2000    0.40  0.005      0.062828         0.062586
  2000    0.05  0.010      0.077034         0.077105
  2000    0.10  0.010      0.074572         0.073833
  2000    0.15  0.010      0.071798         0.071324
  2000    0.20  0.010      0.068470         0.068125
 shots  eta_c_mean  eta_c_sd
  2000    0.059951  0.012655
  5000    0.063443

In [9]:
PHASE_MARGIN_GRID = np.array([0.01, 0.02, 0.03, 0.05], dtype=float)

phase_margin_profile_frames = []
phase_margin_boundary_rows = []

for phase_margin in PHASE_MARGIN_GRID:
    temp = endpoint_df.loc[:, ["eta", "attack", "drift"]].copy()
    psi_values = endpoint_df["psi"].to_numpy(dtype=float)

    temp["p_secure"] = (psi_values > phase_margin).astype(float)
    temp["p_transition"] = (np.abs(psi_values) <= phase_margin).astype(float)
    temp["p_insecure"] = (psi_values < -phase_margin).astype(float)

    grouped = (
        temp.groupby(["drift", "attack", "eta"], as_index=False)[["p_secure", "p_transition", "p_insecure"]]
        .mean()
        .sort_values(["drift", "attack", "eta"])
        .reset_index(drop=True)
    )
    grouped["phase_margin"] = float(phase_margin)
    phase_margin_profile_frames.append(grouped)

    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            sub = grouped[
                (grouped["drift"] == float(drift)) &
                (grouped["attack"] == float(attack))
            ].sort_values("eta")

            eta_values = sub["eta"].to_numpy(dtype=float)
            p_secure_values = sub["p_secure"].to_numpy(dtype=float)

            eta_c = interpolate_crossing(eta_values, p_secure_values, target=0.5)
            dp_deta = susceptibility(p_secure_values, eta_values)

            phase_margin_boundary_rows.append(
                {
                    "phase_margin": float(phase_margin),
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_psecure50": eta_c,
                    "max_abs_dp_deta": float(np.max(np.abs(dp_deta))),
                    "eta_at_max_abs_dp_deta": float(eta_values[int(np.argmax(np.abs(dp_deta)))]),
                }
            )

phase_margin_profiles_df = pd.concat(phase_margin_profile_frames, ignore_index=True)
phase_margin_boundary_df = (
    pd.DataFrame(phase_margin_boundary_rows)
    .sort_values(["phase_margin", "drift", "attack"])
    .reset_index(drop=True)
)

phase_margin_profiles_csv = TAB_DIR / "phase_margin_profiles.csv"
phase_margin_boundary_csv = TAB_DIR / "phase_margin_boundary_summary.csv"

phase_margin_profiles_df.to_csv(phase_margin_profiles_csv, index=False)
phase_margin_boundary_df.to_csv(phase_margin_boundary_csv, index=False)

baseline_margin = 0.02
baseline = (
    phase_margin_boundary_df[phase_margin_boundary_df["phase_margin"] == baseline_margin]
    .loc[:, ["attack", "drift", "eta_c_psecure50"]]
    .rename(columns={"eta_c_psecure50": "eta_c_baseline"})
)

delta_df = phase_margin_boundary_df.merge(baseline, on=["attack", "drift"], how="left")
delta_df["eta_c_delta"] = delta_df["eta_c_psecure50"] - delta_df["eta_c_baseline"]

delta_summary = (
    delta_df.groupby("phase_margin", as_index=False)
    .agg(
        mean_eta_c=("eta_c_psecure50", "mean"),
        mean_delta=("eta_c_delta", "mean"),
        sd_delta=("eta_c_delta", "std"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

selected_drift = 0.010
for i, phase_margin in enumerate(PHASE_MARGIN_GRID):
    sub = phase_margin_boundary_df[
        (phase_margin_boundary_df["phase_margin"] == float(phase_margin)) &
        (phase_margin_boundary_df["drift"] == selected_drift)
    ].sort_values("attack")

    axes[0].plot(
        sub["attack"],
        sub["eta_c_psecure50"],
        label=f"margin={phase_margin:.2f}",
        **series_style(i),
    )

axes[0].set_xlabel("Attack pressure")
axes[0].set_ylabel(r"Estimated boundary $\eta_c$")
axes[0].set_title(r"Boundary sensitivity at fixed drift ($d=0.010$)")
axes[0].legend()

axes[1].errorbar(
    delta_summary["phase_margin"],
    delta_summary["mean_delta"],
    yerr=delta_summary["sd_delta"],
    capsize=4,
    ecolor=series_color(0),
    **series_style(0),
)
axes[1].axhline(0.0, color="#000000", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Phase margin")
axes[1].set_ylabel(r"Boundary shift relative to baseline $\Delta \eta_c$")
axes[1].set_title("Global boundary sensitivity to phase-margin choice")

plt.tight_layout()

phase_margin_fig = FIG_DIR / "phase_margin_sensitivity.png"
plt.savefig(phase_margin_fig, dpi=240)
plt.close()

print("[MARGIN] profile rows =", len(phase_margin_profiles_df))
print("[MARGIN] boundary rows =", len(phase_margin_boundary_df))
print("[TABLE] saved:", phase_margin_profiles_csv)
print("[TABLE] saved:", phase_margin_boundary_csv)
print("[FIGURE] saved:", phase_margin_fig)
print(phase_margin_boundary_df.head(12).to_string(index=False))
print(delta_summary.to_string(index=False))

[MARGIN] profile rows = 10880
[MARGIN] boundary rows = 128
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\phase_margin_profiles.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\phase_margin_boundary_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\phase_margin_sensitivity.png
 phase_margin  attack  drift  eta_c_psecure50  max_abs_dp_deta  eta_at_max_abs_dp_deta
         0.01    0.05  0.005         0.087596            106.4                  0.0875
         0.01    0.10  0.005         0.084464            112.0                  0.0850
         0.01    0.15  0.005         0.082267            123.2                  0.0825
         0.01    0.20  0.005         0.079109            120.0                  0.0800
         0.01    0.25  0.005         0.076600            112.0                  0.0775
         0.01    0.30  0.005         0.073900            107.2                  0.0725
         0.01    0.

In [10]:
SCENARIO_REPEATS = 150

scenario_seed_rng = np.random.default_rng(SEED + 7000)
trajectory_frames = []
trial_rows = []

for sc in SCENARIOS:
    seeds = scenario_seed_rng.integers(0, 2**32 - 1, size=SCENARIO_REPEATS)

    for repeat_id, rep_seed in enumerate(seeds):
        df = simulate_trajectory(
            eta=float(sc["eta"]),
            attack=float(sc["attack"]),
            drift=float(sc["drift"]),
            shots=DEFAULT_SHOTS,
            time_horizon=TIME_HORIZON,
            seed=int(rep_seed),
        ).copy()

        df["scenario"] = sc["name"]
        df["repeat"] = int(repeat_id)
        trajectory_frames.append(df)

        trial_rows.append(
            {
                "scenario": sc["name"],
                "repeat": int(repeat_id),
                "eta": float(sc["eta"]),
                "attack": float(sc["attack"]),
                "drift": float(sc["drift"]),
                "first_transition_time": first_transition_time(df["psi"].values),
                "final_qber_obs": float(df["qber_obs"].iloc[-1]),
                "final_secrecy_margin": float(df["secrecy_margin"].iloc[-1]),
                "final_psi": float(df["psi"].iloc[-1]),
                "final_phase": str(df["phase"].iloc[-1]),
            }
        )

scenario_traj_df = pd.concat(trajectory_frames, ignore_index=True)
scenario_trial_df = pd.DataFrame(trial_rows)

scenario_time_summary_df = (
    scenario_traj_df
    .groupby(["scenario", "t"], as_index=False)
    .agg(
        qber_mean=("qber_obs", "mean"),
        qber_q05=("qber_obs", lambda x: float(np.quantile(x, 0.05))),
        qber_q50=("qber_obs", lambda x: float(np.quantile(x, 0.50))),
        qber_q95=("qber_obs", lambda x: float(np.quantile(x, 0.95))),
        margin_mean=("secrecy_margin", "mean"),
        margin_q05=("secrecy_margin", lambda x: float(np.quantile(x, 0.05))),
        margin_q50=("secrecy_margin", lambda x: float(np.quantile(x, 0.50))),
        margin_q95=("secrecy_margin", lambda x: float(np.quantile(x, 0.95))),
        psi_mean=("psi", "mean"),
        psi_q05=("psi", lambda x: float(np.quantile(x, 0.05))),
        psi_q50=("psi", lambda x: float(np.quantile(x, 0.50))),
        psi_q95=("psi", lambda x: float(np.quantile(x, 0.95))),
        p_secure=("phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
        p_transition=("phase", lambda s: float(np.mean(np.asarray(s) == "transition"))),
        p_insecure=("phase", lambda s: float(np.mean(np.asarray(s) == "insecure"))),
    )
)

def median_transition_time(values):
    values = np.asarray(values, dtype=int)
    valid = values[values >= 0]
    if len(valid) == 0:
        return -1.0
    return float(np.median(valid))

scenario_summary_df = (
    scenario_trial_df
    .groupby("scenario", as_index=False)
    .agg(
        eta=("eta", "first"),
        attack=("attack", "first"),
        drift=("drift", "first"),
        final_qber_mean=("final_qber_obs", "mean"),
        final_qber_sd=("final_qber_obs", "std"),
        final_margin_mean=("final_secrecy_margin", "mean"),
        final_margin_sd=("final_secrecy_margin", "std"),
        final_psi_mean=("final_psi", "mean"),
        final_psi_sd=("final_psi", "std"),
        p_final_secure=("final_phase", lambda s: float(np.mean(np.asarray(s) == "secure"))),
        p_final_transition=("final_phase", lambda s: float(np.mean(np.asarray(s) == "transition"))),
        p_final_insecure=("final_phase", lambda s: float(np.mean(np.asarray(s) == "insecure"))),
        p_ever_transition=("first_transition_time", lambda x: float(np.mean(np.asarray(x) >= 0))),
        median_first_transition_time=("first_transition_time", median_transition_time),
    )
)

scenario_order = [sc["name"] for sc in SCENARIOS]
scenario_summary_df["scenario"] = pd.Categorical(
    scenario_summary_df["scenario"],
    categories=scenario_order,
    ordered=True,
)
scenario_summary_df = scenario_summary_df.sort_values("scenario").reset_index(drop=True)

scenario_time_summary_df["scenario"] = pd.Categorical(
    scenario_time_summary_df["scenario"],
    categories=scenario_order,
    ordered=True,
)
scenario_time_summary_df = scenario_time_summary_df.sort_values(["scenario", "t"]).reset_index(drop=True)

scenario_traj_csv = TAB_DIR / "scenario_trajectory_replicates.csv"
scenario_time_summary_csv = TAB_DIR / "scenario_time_summary.csv"
scenario_summary_csv = TAB_DIR / "scenario_summary.csv"

scenario_traj_df.to_csv(scenario_traj_csv, index=False)
scenario_time_summary_df.to_csv(scenario_time_summary_csv, index=False)
scenario_summary_df.to_csv(scenario_summary_csv, index=False)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True)
for ax, scenario_name in zip(axes.flat, scenario_order):
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario_name]
    ax.plot(sub["t"], sub["psi_mean"], **series_style(0, lw=2.0, marker=False))
    ax.fill_between(sub["t"], sub["psi_q05"], sub["psi_q95"],
                    color=series_color(0), alpha=BAND_ALPHA)
    ax.axhline(0.0, color="#000000", linestyle="--", linewidth=1.2)
    ax.set_title(scenario_name.replace("_", " "))
    ax.set_xlabel("Time")
    ax.set_ylabel(r"Normalized margin $\psi$")
plt.tight_layout()

scenario_psi_fig = FIG_DIR / "scenario_psi_profiles.png"
plt.savefig(scenario_psi_fig, dpi=240)
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True)
for ax, scenario_name in zip(axes.flat, scenario_order):
    sub = scenario_time_summary_df[scenario_time_summary_df["scenario"] == scenario_name]
    ax.plot(sub["t"], sub["qber_mean"], **series_style(1, lw=2.0, marker=False))
    ax.fill_between(sub["t"], sub["qber_q05"], sub["qber_q95"],
                    color=series_color(1), alpha=BAND_ALPHA)
    ax.axhline(QBER_SECURITY_THRESHOLD, color="#000000", linestyle="--",
               linewidth=1.2)
    ax.set_title(scenario_name.replace("_", " "))
    ax.set_xlabel("Time")
    ax.set_ylabel("Observed QBER")
plt.tight_layout()

scenario_qber_fig = FIG_DIR / "scenario_qber_profiles.png"
plt.savefig(scenario_qber_fig, dpi=240)
plt.close()

print("[SCENARIO] replicate rows =", len(scenario_traj_df))
print("[SCENARIO] trial rows =", len(scenario_trial_df))
print("[TABLE] saved:", scenario_traj_csv)
print("[TABLE] saved:", scenario_time_summary_csv)
print("[TABLE] saved:", scenario_summary_csv)
print("[FIGURE] saved:", scenario_psi_fig)
print("[FIGURE] saved:", scenario_qber_fig)
print(scenario_summary_df.to_string(index=False))

[SCENARIO] replicate rows = 72000
[SCENARIO] trial rows = 600
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\scenario_trajectory_replicates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\scenario_time_summary.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\scenario_summary.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\scenario_psi_profiles.png
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\scenario_qber_profiles.png
         scenario   eta  attack  drift  final_qber_mean  final_qber_sd  final_margin_mean  final_margin_sd  final_psi_mean  final_psi_sd  p_final_secure  p_final_transition  p_final_insecure  p_ever_transition  median_first_transition_time
  low_disturbance 0.035    0.08  0.008         0.056422       0.002339           0.336888         0.018990        3.062618      0.172641        1.000000            0.000000               0.

In [ ]:
EW_WINDOW = 15           # W: observation window length, in evaluation windows
EW_SMOOTH = 5            # moving-average length applied to psi within the window
EW_TRAJ_PER_CELL = 200   # trajectories per offset per attack-drift condition

EW_OFFSETS = np.array([-0.008, -0.004, -0.002, 0.000, 0.002, 0.004, 0.008])
EW_WITHIN_OFFSET = -0.004

ew_rng = np.random.default_rng(SEED + 9000)


def simulate_psi_paths(eta, attack, drift, n_traj, shots=DEFAULT_SHOTS,
                       time_horizon=TIME_HORIZON, rng=None):
    """Vectorized generator of normalized-indicator paths.

    Mathematically identical to repeated calls of simulate_trajectory: the
    observed QBER at each evaluation window is an independent binomial draw at
    the deterministic mean for that window, so the whole (n_traj, T) block can
    be drawn at once. Returns an array of shape (n_traj, time_horizon).
    """
    rng = np.random.default_rng(SEED) if rng is None else rng

    qber_mean = np.array(
        [
            deterministic_qber_mean(eta=eta, attack=attack, drift=drift,
                                    t=k, time_horizon=time_horizon)
            for k in range(time_horizon)
        ],
        dtype=float,
    )
    n_eff = effective_detection_count(shots, attack=attack, drift=drift)

    errors = rng.binomial(
        n=n_eff,
        p=np.clip(qber_mean, EPS, 1.0 - EPS),
        size=(int(n_traj), int(time_horizon)),
    )
    qber_obs = errors / float(n_eff)

    margin = (
        1.0
        - 2.0 * binary_entropy(np.minimum(qber_obs, 0.499999))
        - COEFFS["kappa"] / np.sqrt(max(n_eff, 1))
        - COEFFS["w_M"] * float(drift)
    )
    return margin / max(abs(QBER_SECURITY_THRESHOLD), EPS)


def window_min_smoothed(psi_paths, window=EW_WINDOW, smooth=EW_SMOOTH):
    """Test statistic Z of Eq. (11), plus the smoothed series it minimizes.

    Z is the minimum over the observation window of a moving average of length
    `smooth`, the average being taken over available indices near the start of
    the record.
    """
    block = np.asarray(psi_paths, dtype=float)[:, :window]
    csum = np.cumsum(block, axis=1)
    ma = np.empty_like(block)
    for t in range(window):
        lo = max(0, t - smooth + 1)
        total = csum[:, t] - (csum[:, lo - 1] if lo > 0 else 0.0)
        ma[:, t] = total / float(t - lo + 1)
    return ma.min(axis=1), ma


def crossing_index(psi_paths):
    """First evaluation window at which psi <= 0; -1 if it never crosses."""
    crossed = np.asarray(psi_paths, dtype=float) <= 0.0
    any_cross = crossed.any(axis=1)
    idx = np.where(any_cross, crossed.argmax(axis=1), -1)
    return idx


def roc_auc(scores, labels):
    """Rank-based (Mann-Whitney) area under the ROC curve.

    Higher `scores` indicate the positive class. Returns NaN when either class
    is absent.
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n_pos = int((labels == 1).sum())
    n_neg = int((labels == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=float)
    sorted_scores = scores[order]
    i = 0
    while i < len(scores):
        j = i
        while j + 1 < len(scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def roc_curve_from_scores(scores, labels, n_points=512):
    """False-alarm rate, detection rate and threshold psi_al along the ROC.

    The alarm rule is Z <= psi_al, so the score used for ranking is -Z and the
    thresholds returned are in the original psi_al units.
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    thresholds = np.unique(np.quantile(scores, np.linspace(0.0, 1.0, n_points)))
    thresholds = np.concatenate(([-np.inf], thresholds, [np.inf]))
    pos = labels == 1
    neg = labels == 0
    det = np.array([float((scores[pos] <= th).mean()) for th in thresholds])
    far = np.array([float((scores[neg] <= th).mean()) for th in thresholds])
    return far, det, thresholds


ew_rows = []
ew_condition_frames = []
ma_blocks = []          # smoothed in-window series, retained trajectories only
tcross_blocks = []      # crossing index, retained trajectories only

for drift in DRIFT_GRID:
    for attack in ATTACK_GRID:
        boundary_row = boundary_df[
            (boundary_df["attack"] == float(attack)) &
            (boundary_df["drift"] == float(drift))
        ].iloc[0]
        eta_c = float(boundary_row["eta_c_psecure50"])
        if not np.isfinite(eta_c):
            continue

        for offset in EW_OFFSETS:
            eta = float(eta_c + offset)
            psi_paths = simulate_psi_paths(
                eta=eta,
                attack=float(attack),
                drift=float(drift),
                n_traj=EW_TRAJ_PER_CELL,
                shots=DEFAULT_SHOTS,
                rng=ew_rng,
            )

            z_stat, ma = window_min_smoothed(psi_paths)
            t_cross = crossing_index(psi_paths)

            # Genuine prediction: exclude trajectories that had already crossed
            # inside the observation window.
            already = (t_cross >= 0) & (t_cross < EW_WINDOW)
            keep = ~already
            label = (t_cross >= EW_WINDOW).astype(int)

            ma_blocks.append(ma[keep])
            tcross_blocks.append(t_cross[keep])

            ew_condition_frames.append(
                pd.DataFrame(
                    {
                        "attack": float(attack),
                        "drift": float(drift),
                        "eta_c": eta_c,
                        "offset": float(offset),
                        "eta": eta,
                        "z_stat": z_stat[keep],
                        "t_cross": t_cross[keep],
                        "label": label[keep],
                    }
                )
            )

            ew_rows.append(
                {
                    "attack": float(attack),
                    "drift": float(drift),
                    "offset": float(offset),
                    "eta": eta,
                    "n_generated": int(EW_TRAJ_PER_CELL),
                    "n_excluded_early_crossing": int(already.sum()),
                    "n_retained": int(keep.sum()),
                    "n_positive": int(label[keep].sum()),
                    "n_negative": int((label[keep] == 0).sum()),
                }
            )

ew_df = pd.concat(ew_condition_frames, ignore_index=True)
ew_design_df = pd.DataFrame(ew_rows)
ma_all = np.vstack(ma_blocks)
tcross_all = np.concatenate(tcross_blocks)

n_total_generated = int(ew_design_df["n_generated"].sum())
n_analysed = len(ew_df)
n_pos = int((ew_df["label"] == 1).sum())
n_neg = int((ew_df["label"] == 0).sum())

# --- Pooled design ----------------------------------------------------------
# The alarm fires when Z <= psi_al, so -Z is the score for which larger values
# indicate the positive class.

pooled_auc = roc_auc(-ew_df["z_stat"].to_numpy(), ew_df["label"].to_numpy())
far_curve, det_curve, thr_curve = roc_curve_from_scores(
    ew_df["z_stat"].to_numpy(), ew_df["label"].to_numpy()
)

roc_df = pd.DataFrame({"psi_al": thr_curve, "false_alarm_rate": far_curve,
                       "detection_rate": det_curve})


def operating_point_at_far(target_far):
    """Largest threshold whose false-alarm rate does not exceed target_far."""
    feasible = roc_df[roc_df["false_alarm_rate"] <= target_far]
    if feasible.empty:
        return None
    return feasible.loc[feasible["detection_rate"].idxmax()]


def operating_point_at_dr(target_dr):
    """Smallest threshold achieving at least target_dr detection rate."""
    feasible = roc_df[roc_df["detection_rate"] >= target_dr]
    if feasible.empty:
        return None
    return feasible.loc[feasible["false_alarm_rate"].idxmin()]


operating_rows = []
for target in (0.01, 0.05, 0.10):
    row = operating_point_at_far(target)
    if row is not None:
        operating_rows.append(
            {
                "criterion": f"FAR <= {target:.2f}",
                "psi_al": float(row["psi_al"]),
                "detection_rate": float(row["detection_rate"]),
                "false_alarm_rate": float(row["false_alarm_rate"]),
            }
        )
for target in (0.90, 0.95, 0.99):
    row = operating_point_at_dr(target)
    if row is not None:
        operating_rows.append(
            {
                "criterion": f"DR >= {target:.2f}",
                "psi_al": float(row["psi_al"]),
                "detection_rate": float(row["detection_rate"]),
                "false_alarm_rate": float(row["false_alarm_rate"]),
            }
        )

ew_operating_df = pd.DataFrame(operating_rows)

psi_al_5pct = float(ew_operating_df.loc[
    ew_operating_df["criterion"] == "FAR <= 0.05", "psi_al"].iloc[0])

alarmed = ma_all <= psi_al_5pct
t_alarm_all = np.where(alarmed.any(axis=1), alarmed.argmax(axis=1), -1)

lead_valid = (ew_df["label"].to_numpy() == 1) & (t_alarm_all >= 0)
lead_times = (tcross_all[lead_valid] - t_alarm_all[lead_valid]).astype(float)

lead_median = float(np.median(lead_times)) if len(lead_times) else np.nan
lead_q25 = float(np.quantile(lead_times, 0.25)) if len(lead_times) else np.nan
lead_q75 = float(np.quantile(lead_times, 0.75)) if len(lead_times) else np.nan

# --- Within-condition design ------------------------------------------------
# Between-condition differences are eliminated by construction: the AUC is
# recomputed separately within each fixed (eta, a, d) stratum at the common
# offset eta_c - 0.004.

within_rows = []
within_sub = ew_df[np.isclose(ew_df["offset"], EW_WITHIN_OFFSET)]
for (attack, drift), grp in within_sub.groupby(["attack", "drift"]):
    auc = roc_auc(-grp["z_stat"].to_numpy(), grp["label"].to_numpy())
    within_rows.append(
        {
            "attack": float(attack),
            "drift": float(drift),
            "n": int(len(grp)),
            "n_positive": int((grp["label"] == 1).sum()),
            "n_negative": int((grp["label"] == 0).sum()),
            "auc": auc,
        }
    )

ew_within_df = pd.DataFrame(within_rows)
within_valid = ew_within_df["auc"].dropna().to_numpy()
within_mean = float(np.mean(within_valid)) if len(within_valid) else np.nan
within_sd = float(np.std(within_valid, ddof=1)) if len(within_valid) > 1 else np.nan

# --- Persist ----------------------------------------------------------------

ew_design_csv = TAB_DIR / "early_warning_design.csv"
ew_roc_csv = TAB_DIR / "early_warning_roc.csv"
ew_operating_csv = TAB_DIR / "table7_early_warning_operating_points.csv"
ew_within_csv = TAB_DIR / "early_warning_within_condition_auc.csv"

ew_design_df.to_csv(ew_design_csv, index=False)
roc_df.to_csv(ew_roc_csv, index=False)
ew_operating_df.to_csv(ew_operating_csv, index=False)
ew_within_df.to_csv(ew_within_csv, index=False)

# --- Figure 7 ---------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4))

finite = np.isfinite(far_curve) & np.isfinite(det_curve)
axes[0].plot(far_curve[finite], det_curve[finite],
             **series_style(0, lw=2.0, marker=False))
axes[0].plot([0, 1], [0, 1], color="#000000", linestyle=":", linewidth=1.2,
             label="chance")
op5 = ew_operating_df[ew_operating_df["criterion"] == "FAR <= 0.05"].iloc[0]
axes[0].plot([op5["false_alarm_rate"]], [op5["detection_rate"]],
             color=OKABE_ITO[1], marker="D", markersize=9, linestyle="none",
             label=f"5% FAR: DR={op5['detection_rate']:.4f}")
axes[0].set_xlabel("False-alarm rate")
axes[0].set_ylabel("Detection rate")
axes[0].set_title(f"(a) Pooled ROC (AUC = {pooled_auc:.4f})")
axes[0].legend(loc="lower right", fontsize=9)

display_lead = lead_times[lead_times <= 50]
axes[1].hist(display_lead, bins=np.arange(0, 52, 2),
             color=OKABE_ITO[2], edgecolor="black", linewidth=0.6)
axes[1].axvline(lead_median, color=OKABE_ITO[1], linestyle="--", linewidth=1.8,
                label=f"median = {lead_median:.0f}")
axes[1].set_xlabel("Warning lead time (evaluation windows)")
axes[1].set_ylabel("Count")
axes[1].set_title("(b) Lead time at 5% false-alarm rate")
axes[1].legend(fontsize=9)

axes[2].hist(within_valid, bins=np.linspace(0.30, 0.70, 21),
             color=OKABE_ITO[5], edgecolor="black", linewidth=0.6)
axes[2].axvline(0.5, color="#000000", linestyle=":", linewidth=1.5,
                label="chance (0.5)")
axes[2].axvline(within_mean, color=OKABE_ITO[1], linestyle="--", linewidth=1.8,
                label=f"mean = {within_mean:.4f}")
axes[2].set_xlabel("Within-condition AUC")
axes[2].set_ylabel("Count")
axes[2].set_title(r"(c) AUC within fixed condition ($\eta_c - 0.004$)")
axes[2].legend(fontsize=9)

plt.tight_layout()
ew_fig = FIG_DIR / "fig7_early_warning_validation.png"
plt.savefig(ew_fig, dpi=240)
plt.close()

print("[EW] trajectories generated =", n_total_generated)
print("[EW] trajectories analysed  =", n_analysed,
      f"(crossed: {n_pos}, did not cross: {n_neg})")
print("[EW] pooled AUC =", round(pooled_auc, 4))
print("[EW] lead time at 5% FAR: median =", lead_median,
      f"(IQR {lead_q25:.0f}-{lead_q75:.0f})")
print("[EW] within-condition AUC =", round(within_mean, 4), "+/-",
      round(within_sd, 4),
      f"(range {within_valid.min():.3f}-{within_valid.max():.3f})")
print("[TABLE] saved:", ew_operating_csv)
print("[FIGURE] saved:", ew_fig)
print()
print("[TABLE 7] Early-warning operating points")
print(ew_operating_df.to_string(index=False))


[EW] trajectories generated = 44800
[EW] trajectories analysed  = 26697 (crossed: 18705, did not cross: 7992)
[EW] pooled AUC = 0.9562
[EW] lead time at 5% FAR: median = 17.0 (IQR 13-21)
[EW] within-condition AUC = 0.5172 +/- 0.0535 (range 0.404-0.619)
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table7_early_warning_operating_points.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\fig7_early_warning_validation.png

[TABLE 7] Early-warning operating points
  criterion   psi_al  detection_rate  false_alarm_rate
FAR <= 0.01 0.392097        0.540444          0.009885
FAR <= 0.05 0.473287        0.746913          0.049550
FAR <= 0.10 0.517419        0.842716          0.099850
 DR >= 0.90 0.552094        0.900027          0.142267
 DR >= 0.95 0.593423        0.951190          0.192442
 DR >= 0.99 0.694057        0.990056          0.343343


In [ ]:
PERTURBATION = 0.10          # +/-10% perturbation of each structural coefficient
JOINT_DRAWS = 2000           # replicates in the joint analysis

COEFF_ROLES = {
    "w_d": "Drift weight, mean QBER, Eq. (2)",
    "w_a": "Attack weight, mean QBER, Eq. (2)",
    "kappa": "Finite-key penalty, Eq. (6)",
    "w_M": "Drift term, secrecy margin, Eq. (6)",
    "r_d": "Drift term, retention, Eq. (3)",
    "r_a": "Attack term, retention, Eq. (3)",
}

sens_rng = np.random.default_rng(SEED + 11000)


def compute_probability_boundaries(rng):
    """Probability-based boundary for every attack-drift condition.

    Re-executes the full stochastic endpoint pipeline under whatever structural
    coefficients are currently set in COEFFS, and returns a DataFrame with one
    row per (attack, drift) condition.
    """
    rows = []
    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            p_secure = np.empty(len(ETA_GRID), dtype=float)
            for i, eta in enumerate(ETA_GRID):
                df = simulate_endpoint_replicates(
                    eta=float(eta),
                    attack=float(attack),
                    drift=float(drift),
                    shots=DEFAULT_SHOTS,
                    n_repeats=N_REPEATS,
                    rng=rng,
                )
                p_secure[i] = float((df["phase"] == "secure").mean())
            rows.append(
                {
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_psecure50": interpolate_crossing(
                        ETA_GRID, p_secure, target=0.5
                    ),
                }
            )
    return pd.DataFrame(rows)


def deterministic_condition_boundaries(coeffs, shots=DEFAULT_SHOTS,
                                       t_eval=TIME_HORIZON - 1):
    """Margin-based boundary per attack-drift condition, with no sampling noise.

    Evaluates the mean-margin zero crossing analytically under the supplied
    coefficient dictionary. Used to report a Monte-Carlo-free companion to the
    stochastic one-at-a-time sensitivities: for coefficients whose true effect
    is far below the replicate noise floor, the stochastic estimate measures
    the noise floor rather than the effect.
    """
    seasonal = 0.006 * np.sin(2.0 * np.pi * t_eval / max(TIME_HORIZON, 1))
    trend = 0.010 * (t_eval / max(TIME_HORIZON - 1, 1))

    rows = []
    for drift in DRIFT_GRID:
        for attack in ATTACK_GRID:
            qber = np.clip(
                ETA_GRID + coeffs["w_a"] * attack + coeffs["w_d"] * drift
                + seasonal + trend,
                0.0, 1.0,
            )
            retention = float(
                np.clip(1.0 - coeffs["r_a"] * attack - coeffs["r_d"] * drift, 0.35, 1.0)
            )
            n_eff = max(50.0, round(shots * retention))
            margin = (
                1.0
                - 2.0 * binary_entropy(np.minimum(qber, 0.499999))
                - coeffs["kappa"] / np.sqrt(n_eff)
                - coeffs["w_M"] * drift
            )
            rows.append(
                {
                    "attack": float(attack),
                    "drift": float(drift),
                    "eta_c_det": interpolate_crossing(ETA_GRID, margin, target=0.0),
                }
            )
    return pd.DataFrame(rows)


# --- One-at-a-time analysis -------------------------------------------------
# Each coefficient is perturbed by +/-10% with the others held at baseline and
# the full stochastic endpoint pipeline is re-executed. The reported
# sensitivity is the mean absolute relative change in the probability-based
# boundary, averaged over all 32 attack-drift conditions and over both signs.

reset_coefficients()
oat_baseline = compute_probability_boundaries(np.random.default_rng(SEED + 11001))
oat_baseline = oat_baseline.rename(columns={"eta_c_psecure50": "eta_c_base"})

oat_rows = []
oat_detail_rows = []

det_baseline = deterministic_condition_boundaries(COEFF_BASELINE).rename(
    columns={"eta_c_det": "eta_c_det_base"}
)

for coeff_name in COEFF_ROLES:
    base_value = COEFF_BASELINE[coeff_name]
    rel_changes = []
    det_rel_changes = []

    for sign in (+1.0, -1.0):
        reset_coefficients()
        COEFFS[coeff_name] = base_value * (1.0 + sign * PERTURBATION)

        perturbed = compute_probability_boundaries(
            np.random.default_rng(SEED + 11001)
        )
        merged = perturbed.merge(oat_baseline, on=["attack", "drift"], how="inner")
        rel = np.abs(
            (merged["eta_c_psecure50"] - merged["eta_c_base"]) / merged["eta_c_base"]
        )
        rel = rel.replace([np.inf, -np.inf], np.nan).dropna()
        rel_changes.append(float(rel.mean()))

        det_perturbed = deterministic_condition_boundaries(COEFFS)
        det_merged = det_perturbed.merge(det_baseline, on=["attack", "drift"])
        det_rel = np.abs(
            (det_merged["eta_c_det"] - det_merged["eta_c_det_base"])
            / det_merged["eta_c_det_base"]
        )
        det_rel = det_rel.replace([np.inf, -np.inf], np.nan).dropna()
        det_rel_changes.append(float(det_rel.mean()))

        oat_detail_rows.append(
            {
                "coefficient": coeff_name,
                "sign": "+10%" if sign > 0 else "-10%",
                "value": float(COEFFS[coeff_name]),
                "mean_abs_rel_change_pct": float(rel.mean() * 100.0),
                "mean_abs_rel_change_pct_deterministic": float(det_rel.mean() * 100.0),
                "mean_boundary": float(perturbed["eta_c_psecure50"].mean()),
            }
        )

    reset_coefficients()
    oat_rows.append(
        {
            "coefficient": coeff_name,
            "baseline": base_value,
            "role": COEFF_ROLES[coeff_name],
            "sensitivity_pct": float(np.mean(rel_changes) * 100.0),
            "sensitivity_pct_deterministic": float(np.mean(det_rel_changes) * 100.0),
        }
    )

reset_coefficients()

oat_df = (
    pd.DataFrame(oat_rows)
    .sort_values("sensitivity_pct", ascending=False)
    .reset_index(drop=True)
)
oat_detail_df = pd.DataFrame(oat_detail_rows)

# Monte Carlo noise floor: the apparent sensitivity of an unperturbed rerun.
# Any stochastic sensitivity at or below this level is measuring replicate
# noise rather than the coefficient's effect.
reset_coefficients()
noise_check = compute_probability_boundaries(np.random.default_rng(SEED + 11002))
noise_merged = noise_check.merge(oat_baseline, on=["attack", "drift"])
noise_rel = np.abs(
    (noise_merged["eta_c_psecure50"] - noise_merged["eta_c_base"])
    / noise_merged["eta_c_base"]
).replace([np.inf, -np.inf], np.nan).dropna()
oat_noise_floor_pct = float(noise_rel.mean() * 100.0)


def deterministic_mean_boundary_vectorized(draws, shots=DEFAULT_SHOTS,
                                           t_eval=TIME_HORIZON - 1):
    """Mean margin-based boundary across conditions, for each coefficient draw.

    `draws` maps coefficient name -> array of shape (n_draws,). Returns an
    array of shape (n_draws,) holding the boundary averaged over the 32
    attack-drift conditions.
    """
    n_draws = len(next(iter(draws.values())))
    conditions = [(a, d) for d in DRIFT_GRID for a in ATTACK_GRID]

    seasonal = 0.006 * np.sin(2.0 * np.pi * t_eval / max(TIME_HORIZON, 1))
    trend = 0.010 * (t_eval / max(TIME_HORIZON - 1, 1))

    eta = ETA_GRID[None, :]                      # (1, n_eta)
    boundaries = np.full((n_draws, len(conditions)), np.nan, dtype=float)

    for c, (attack, drift) in enumerate(conditions):
        w_a = draws["w_a"][:, None]
        w_d = draws["w_d"][:, None]
        r_a = draws["r_a"][:, None]
        r_d = draws["r_d"][:, None]
        kappa = draws["kappa"][:, None]
        w_M = draws["w_M"][:, None]

        qber = np.clip(eta + w_a * attack + w_d * drift + seasonal + trend, 0.0, 1.0)

        retention = np.clip(1.0 - r_a * attack - r_d * drift, 0.35, 1.0)
        n_eff = np.maximum(50.0, np.round(shots * retention))

        margin = (
            1.0
            - 2.0 * binary_entropy(np.minimum(qber, 0.499999))
            - kappa / np.sqrt(n_eff)
            - w_M * drift
        )                                        # (n_draws, n_eta)

        # Vectorized first zero crossing along the eta axis.
        sign_change = (margin[:, :-1] * margin[:, 1:]) < 0.0
        has_cross = sign_change.any(axis=1)
        first = np.where(has_cross, sign_change.argmax(axis=1), 0)

        rows = np.arange(n_draws)
        y0 = margin[rows, first]
        y1 = margin[rows, first + 1]
        x0 = ETA_GRID[first]
        x1 = ETA_GRID[first + 1]
        crossing = x0 - y0 * (x1 - x0) / (y1 - y0)

        boundaries[:, c] = np.where(has_cross, crossing, np.nan)

    return np.nanmean(boundaries, axis=1)


joint_draws = {
    name: sens_rng.uniform(
        base * (1.0 - PERTURBATION), base * (1.0 + PERTURBATION), size=JOINT_DRAWS
    )
    for name, base in COEFF_BASELINE.items()
}

joint_boundaries = deterministic_mean_boundary_vectorized(joint_draws)

joint_median = float(np.nanmedian(joint_boundaries))
joint_lo = float(np.nanquantile(joint_boundaries, 0.025))
joint_hi = float(np.nanquantile(joint_boundaries, 0.975))
joint_halfwidth_pct = float(100.0 * 0.5 * (joint_hi - joint_lo) / joint_median)

joint_df = pd.DataFrame({"replicate": np.arange(JOINT_DRAWS), "mean_boundary": joint_boundaries})
for name in COEFF_BASELINE:
    joint_df[name] = joint_draws[name]

boundary_valid = boundary_df.dropna(subset=["eta_c_psecure50"])
mean_boundary = float(boundary_valid["eta_c_psecure50"].mean())

boot_halfwidths = 0.5 * (
    boundary_valid["eta_c_boot_high"] - boundary_valid["eta_c_boot_low"]
)
mc_halfwidth_pct = float(100.0 * boot_halfwidths.mean() / mean_boundary)

cond_halfwidth_pct = float(
    100.0
    * 0.5
    * (boundary_valid["eta_c_psecure50"].max() - boundary_valid["eta_c_psecure50"].min())
    / mean_boundary
)

budget_df = pd.DataFrame(
    [
        {"contribution": "Monte Carlo variability (bootstrap)",
         "half_width_pct": mc_halfwidth_pct},
        {"contribution": "Model coefficients (joint +/-10%)",
         "half_width_pct": joint_halfwidth_pct},
        {"contribution": "Operating condition (a, d range)",
         "half_width_pct": cond_halfwidth_pct},
    ]
)

# --- Persist ----------------------------------------------------------------

oat_csv = TAB_DIR / "table8_parameter_sensitivity.csv"
oat_detail_csv = TAB_DIR / "parameter_sensitivity_detail.csv"
joint_csv = TAB_DIR / "parameter_joint_draws.csv"
budget_csv = TAB_DIR / "table8_uncertainty_budget.csv"

oat_df.to_csv(oat_csv, index=False)
oat_detail_df.to_csv(oat_detail_csv, index=False)
joint_df.to_csv(joint_csv, index=False)
budget_df.to_csv(budget_csv, index=False)

# --- Figure 8 ---------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

labels = {
    "w_d": r"$w_d = 0.90$", "w_a": r"$w_a = 0.055$", "kappa": r"$\kappa = 2.6$",
    "w_M": r"$w_M = 1.35$", "r_d": r"$r_d = 1.10$", "r_a": r"$r_a = 0.18$",
}
ypos = np.arange(len(oat_df))
axes[0].barh(
    ypos,
    oat_df["sensitivity_pct"],
    color=[series_color(i) for i in range(len(oat_df))],
    edgecolor="black",
    linewidth=0.7,
)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels([labels[c] for c in oat_df["coefficient"]])
axes[0].invert_yaxis()
axes[0].axvline(oat_noise_floor_pct, color="#000000", linestyle=":", linewidth=1.4)
axes[0].text(oat_noise_floor_pct, -0.75, " Monte Carlo noise floor",
             fontsize=8, va="center", ha="left")
axes[0].set_xlabel("Mean absolute relative change in boundary (%)")
axes[0].set_title(r"(a) One-at-a-time sensitivity ($\pm 10\%$)")
for y, v in zip(ypos, oat_df["sensitivity_pct"]):
    axes[0].text(v, y, f" {v:.2f}", va="center", fontsize=9)

bpos = np.arange(len(budget_df))
axes[1].barh(
    bpos,
    budget_df["half_width_pct"],
    color=[series_color(i + 3) for i in range(len(budget_df))],
    edgecolor="black",
    linewidth=0.7,
    hatch="//",
)
axes[1].set_yticks(bpos)
axes[1].set_yticklabels(
    ["Monte Carlo\n(bootstrap)", "Model coefficients\n(joint)", "Operating\ncondition"]
)
axes[1].invert_yaxis()
axes[1].set_xscale("log")
axes[1].set_xlabel("Half-width relative to mean boundary (%), log scale")
axes[1].set_title("(b) Uncertainty budget")
for y, v in zip(bpos, budget_df["half_width_pct"]):
    axes[1].text(v, y, f" {v:.2f}", va="center", fontsize=9)

plt.tight_layout()
sens_fig = FIG_DIR / "fig8_parameter_uncertainty.png"
plt.savefig(sens_fig, dpi=240)
plt.close()

print("[SENS] baseline mean boundary =", round(mean_boundary, 6))
print("[SENS] joint median =", round(joint_median, 6),
      "95% interval = [", round(joint_lo, 6), ",", round(joint_hi, 6), "]",
      "half-width =", round(joint_halfwidth_pct, 2), "%")
print("[TABLE] saved:", oat_csv)
print("[TABLE] saved:", budget_csv)
print("[FIGURE] saved:", sens_fig)
print()
print("[TABLE 8 upper] One-at-a-time sensitivity")
print(
    oat_df[
        ["coefficient", "role", "sensitivity_pct", "sensitivity_pct_deterministic"]
    ].to_string(index=False)
)
print(f"[SENS] Monte Carlo noise floor of the stochastic estimator: "
      f"{oat_noise_floor_pct:.3f}% (unperturbed rerun)")
print()
print("[TABLE 8 lower] Uncertainty budget")
print(budget_df.to_string(index=False))


[SENS] baseline mean boundary = 0.065076
[SENS] joint median = 0.065413 95% interval = [ 0.063213 , 0.067623 ] half-width = 3.37 %
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table8_parameter_sensitivity.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table8_uncertainty_budget.csv
[FIGURE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\figures\fig8_parameter_uncertainty.png

[TABLE 8 upper] One-at-a-time sensitivity
coefficient                                role  sensitivity_pct  sensitivity_pct_deterministic
        w_d    Drift weight, mean QBER, Eq. (2)         2.638697                       2.571580
        w_a   Attack weight, mean QBER, Eq. (2)         2.077805                       2.072067
      kappa         Finite-key penalty, Eq. (6)         0.665140                       0.679795
        w_M Drift term, secrecy margin, Eq. (6)         0.625686                       0.610283
        r_a     Attack term,

In [13]:
boundary_table_df = (
    boundary_df.loc[
        :,
        [
            "attack",
            "drift",
            "eta_c_margin",
            "eta_c_psecure50",
            "eta_c_boot_low",
            "eta_c_boot_med",
            "eta_c_boot_high",
            "max_abs_dp_deta",
        ],
    ]
    .copy()
    .sort_values(["drift", "attack"])
    .reset_index(drop=True)
)

shot_robustness_df = (
    shot_boundary_df.groupby("shots", as_index=False)
    .agg(
        mean_eta_c=("eta_c_psecure50", "mean"),
        sd_eta_c=("eta_c_psecure50", "std"),
        min_eta_c=("eta_c_psecure50", "min"),
        max_eta_c=("eta_c_psecure50", "max"),
    )
    .sort_values("shots")
    .reset_index(drop=True)
)

phase_margin_robustness_df = delta_summary.copy().sort_values("phase_margin").reset_index(drop=True)

early_warning_table_df = ew_operating_df.copy().reset_index(drop=True)

parameter_sensitivity_table_df = oat_df.loc[
    :, ["coefficient", "role", "baseline", "sensitivity_pct"]
].copy().reset_index(drop=True)

uncertainty_budget_table_df = budget_df.copy().reset_index(drop=True)

scenario_table_df = (
    scenario_summary_df.loc[
        :,
        [
            "scenario",
            "eta",
            "attack",
            "drift",
            "final_qber_mean",
            "final_margin_mean",
            "final_psi_mean",
            "p_final_secure",
            "p_final_transition",
            "p_final_insecure",
            "p_ever_transition",
            "median_first_transition_time",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

early_warning_table_csv = TAB_DIR / "table_early_warning_operating_points.csv"
parameter_sensitivity_table_csv = TAB_DIR / "table_parameter_sensitivity.csv"
uncertainty_budget_table_csv = TAB_DIR / "table_uncertainty_budget.csv"

early_warning_table_df.to_csv(early_warning_table_csv, index=False)
parameter_sensitivity_table_df.to_csv(parameter_sensitivity_table_csv, index=False)
uncertainty_budget_table_df.to_csv(uncertainty_budget_table_csv, index=False)

boundary_table_csv = TAB_DIR / "table_boundary_estimates.csv"
shot_robustness_csv = TAB_DIR / "table_shot_count_robustness.csv"
phase_margin_robustness_csv = TAB_DIR / "table_phase_margin_robustness.csv"
scenario_table_csv = TAB_DIR / "table_scenario_summary.csv"

boundary_table_df.to_csv(boundary_table_csv, index=False)
shot_robustness_df.to_csv(shot_robustness_csv, index=False)
phase_margin_robustness_df.to_csv(phase_margin_robustness_csv, index=False)
scenario_table_df.to_csv(scenario_table_csv, index=False)

generated_files = sorted(str(path) for path in PROJECT_ROOT.rglob("*") if path.is_file())

summary_lines = [
    "BB84 finite-key phase study: run summary",
    "=" * 60,
    f"Run ID: {RUN_ID}",
    f"Project root: {PROJECT_ROOT}",
    "",
    "Study scope:",
    "- Operational boundary estimation from repeated endpoint simulations",
    "- Bootstrap uncertainty for secure-state probability boundary",
    "- Robustness checks across shot count and descriptive phase margin",
    "- Repeated stochastic scenario analysis with temporal uncertainty bands",
    "- Quantitative early-warning validation (pooled and within-condition designs)",
    "- Model-parameter sensitivity and a three-way uncertainty budget",
    "",
    "Key tables:",
    f"- {boundary_table_csv}",
    f"- {shot_robustness_csv}",
    f"- {phase_margin_robustness_csv}",
    f"- {scenario_table_csv}",
    f"- {ew_operating_csv}",
    f"- {oat_csv}",
    f"- {budget_csv}",
    "",
    "Key figures:",
    f"- {FIG_DIR / 'phase_boundary_with_uncertainty.png'}",
    f"- {FIG_DIR / 'transition_profiles_and_susceptibility.png'}",
    f"- {FIG_DIR / 'shot_count_boundary_dependence.png'}",
    f"- {FIG_DIR / 'phase_margin_sensitivity.png'}",
    f"- {FIG_DIR / 'scenario_psi_profiles.png'}",
    f"- {FIG_DIR / 'scenario_qber_profiles.png'}",
    f"- {FIG_DIR / 'fig7_early_warning_validation.png'}",
    f"- {FIG_DIR / 'fig8_parameter_uncertainty.png'}",
    "",
    "Interpretation:",
    "This run characterizes an operational secure-to-insecure boundary in a BB84-inspired finite-key model under disturbance, adversarial pressure, and implementation drift. Boundary estimates are obtained from repeated finite-sample simulations, uncertainty is quantified by bootstrap resampling, and robustness is examined with respect to shot count and descriptive phase-margin choice. The normalized indicator is validated as an early-warning quantity under an explicit prediction task, which separates pooled discrimination from within-condition discrimination, and the structural coefficients of the model are perturbed to place sampling uncertainty, coefficient uncertainty, and systematic operating-condition variation on a common scale.",

    f"Early warning: pooled AUC = {pooled_auc:.4f}; within-condition AUC = {within_mean:.4f} +/- {within_sd:.4f} over {len(within_valid)} conditions; median lead time = {lead_median:.0f} evaluation windows at a 5% false-alarm rate; {n_analysed} trajectories analysed ({n_pos} crossed, {n_neg} did not).",
    f"Uncertainty budget: Monte Carlo {mc_halfwidth_pct:.2f}%, model coefficients {joint_halfwidth_pct:.2f}%, operating condition {cond_halfwidth_pct:.2f}%.",
    "",
    "Generated files:",
]
summary_lines.extend(f"- {fp}" for fp in generated_files)

summary_path = OTH_DIR / "study_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("[TABLE] saved:", boundary_table_csv)
print("[TABLE] saved:", shot_robustness_csv)
print("[TABLE] saved:", phase_margin_robustness_csv)
print("[TABLE] saved:", scenario_table_csv)
print("[SUMMARY] saved:", summary_path)

print("\n[BOUNDARY TABLE]")
print(boundary_table_df.head(10).to_string(index=False))

print("\n[SHOT ROBUSTNESS]")
print(shot_robustness_df.to_string(index=False))

print("\n[PHASE-MARGIN ROBUSTNESS]")
print(phase_margin_robustness_df.to_string(index=False))

print("\n[SCENARIO TABLE]")
print(scenario_table_df.to_string(index=False))

print("\n[EARLY-WARNING OPERATING POINTS]")
print(early_warning_table_df.to_string(index=False))

print("\n[PARAMETER SENSITIVITY]")
print(parameter_sensitivity_table_df.to_string(index=False))

print("\n[UNCERTAINTY BUDGET]")
print(uncertainty_budget_table_df.to_string(index=False))

[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table_boundary_estimates.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table_shot_count_robustness.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table_phase_margin_robustness.csv
[TABLE] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\tables\table_scenario_summary.csv
[SUMMARY] saved: Outputs\bb84_finite_key_phase_study_20260824_225821\others\study_summary.txt

[BOUNDARY TABLE]
 attack  drift  eta_c_margin  eta_c_psecure50  eta_c_boot_low  eta_c_boot_med  eta_c_boot_high  max_abs_dp_deta
   0.05  0.005      0.087717         0.087237        0.086738        0.087252         0.087781            104.0
   0.10  0.005      0.084732         0.084412        0.084133        0.084427         0.084922            113.6
   0.15  0.005      0.082298         0.082045        0.081699        0.082055         0.082466            126.4
   0.20  0.005      0.